# DFST Softmax Arithmetic

Compare hidden dimensions and dataset sizes for binary arithmetic.

Install the package with `pip install -e ".[analysis]"` from the repository root. This notebook reads the final evaluations selected in `paper/figures.yaml`. The notebook uses saved results without training or checkpoint evaluation.

Every selected run requires a final evaluation. Zero denotes a measured result; `≥` marks a tested ceiling. The notebook saves task figures under `analysis_outputs/paper_figures/DFST_Softmax/<task>/` and the arithmetic metric table under `DFST_Softmax/arithmetic/`. `DEL_ANALYSIS_DIR` selects another figure root and preserves these subfolders.


In [ ]:
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path(os.environ["DEL_REPO_ROOT"]) if os.environ.get("DEL_REPO_ROOT") else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/differentiable_exact_learning").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from differentiable_exact_learning.analysis.release import figure_group, final_results_table
from differentiable_exact_learning.analysis.figure_data import panel_data
from differentiable_exact_learning.analysis.paper_heatmaps import plot_heatmap_group, save_figure, figure_output_dir
from differentiable_exact_learning.analysis.plot_style import apply_paper_style
apply_paper_style()
OUTPUT_DIR = figure_output_dir(ROOT, 'DFST_Softmax')
SUMMARY_DIR = figure_output_dir(ROOT, 'DFST_Softmax', "arithmetic")
def export(fig, name):
    paths = save_figure(fig, OUTPUT_DIR / name)
    display(fig)
    plt.close(fig)
    return paths

In [ ]:
TASKS = ("add2", "mult2")
records = {task: figure_group(ROOT, f"DFST_Softmax_{task}") for task in TASKS}
results = final_results_table([row for rows in records.values() for row in rows])
display(results)

## Hidden Dimensions and Dataset Size

ULG and RLG retain the saved ceilings and evaluation settings.

In [ ]:
for task in TASKS:
    rows = records[task]
    xs = sorted({r["config"]["model"]["hidden_size"] for r in rows})
    ys = sorted({r["config"]["data"]["size"] for r in rows})
    panels = []
    for metric in ("ulg", "rlg"):
        panel = panel_data(rows, xs, ys, softmax=True, metric=metric, title=metric.upper())
        panel["cap"] = 5000 if task == "add2" else 1000
        panels.append(panel)
        export(plot_heatmap_group([panel], metric.upper(), ncols=1), f"{task}/softmax_{task}_{metric}")
    export(plot_heatmap_group(panels, "Generalization", ncols=2), f"{task}/softmax_{task}_ulg_rlg")
results.to_csv(SUMMARY_DIR / "softmax_arithmetic.csv", index=False)